# 04 · Analisis exploratorio

> **Objetivo:** entender la distribucion de las variables, la temporalidad
> del negocio y las asociaciones entre `Quantity`, `Price` y los agregados
> por cliente. Reproducimos las Figuras 6–9 y las Tablas 4, 5 y 7 del OVA.

## Que vamos a hacer

1. Univariado: cuartiles y cercas de Tukey para `Price` y `Quantity`.
2. Temporal: ventas por mes, mapa dia × hora.
3. Asociaciones: Pearson y Spearman entre `Quantity` y `Price`.
4. Nivel cliente: Spearman frecuencia–monetario y participacion del 20 %
   de mayor valor.

> Si quieres las figuras finales con la paleta del OVA, usa
> `scripts/figuras_u2.py`; aqui nos quedamos con tablas para discutir.

In [1]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
import polars as pl
from segmentacion.config import PROCESSED_DATA_DIR
from segmentacion.data.cargar import leer_parquet
from segmentacion.eda.resumen import (
    asociaciones,
    cercas_tukey,
    cuartiles,
    matriz_dia_hora,
    ventas_por_mes,
)

PARQUET_VENTAS = PROCESSED_DATA_DIR / "ventas_validas.parquet"
assert PARQUET_VENTAS.exists(), "Ejecuta `02_limpieza.ipynb` primero."
ventas = leer_parquet(PARQUET_VENTAS)
print(f"ventas validas: {ventas.height:,}".replace(",", " "))

ventas validas: 779 425


## 1. Univariado — `Price` y `Quantity`

In [3]:
print("Price:", cuartiles(ventas, "Price"))
print("Quantity:", cuartiles(ventas, "Quantity"))

Price: {'min': 0.001, 'Q1': 1.25, 'mediana': 1.95, 'Q3': 3.75, 'max': 10953.5, 'media': 3.2184879853738333}
Quantity: {'min': 1.0, 'Q1': 2.0, 'mediana': 6.0, 'Q3': 12.0, 'max': 80995.0, 'media': 13.489369727683869}


### Cercas de Tukey

El OVA cita: `Price` → cerca superior 7,50 (8,40 % la superan); `Quantity`
→ cerca superior 27 (6,56 %). Son los umbrales que usaremos en U3 para
decidir si "cortar" o "transformar" los outliers.

In [4]:
inf_p, sup_p, frac_p = cercas_tukey(ventas, "Price")
inf_q, sup_q, frac_q = cercas_tukey(ventas, "Quantity")
print(f"Price   -> cerca sup = {sup_p:.2f}; fraccion > sup = {frac_p:.2%}")
print(f"Quantity-> cerca sup = {sup_q:.2f}; fraccion > sup = {frac_q:.2%}")

Price   -> cerca sup = 7.50; fraccion > sup = 8.40%
Quantity-> cerca sup = 27.00; fraccion > sup = 6.56%


## 2. Temporal — ventas por mes y mapa dia × hora

In [5]:
mes = ventas_por_mes(ventas)
mes.tail(6)

mes,ventas
datetime[ms],f64
2011-07-01 00:00:00,598962.901
2011-08-01 00:00:00,644051.04
2011-09-01 00:00:00,950690.202
2011-10-01 00:00:00,1.0356e6
2011-11-01 00:00:00,1.1562e6
2011-12-01 00:00:00,517208.44


In [6]:
hm = matriz_dia_hora(ventas)
# Pivot a matriz 7 x H para inspeccion visual rapida.
hm.pivot(index="dia", on="hora", values="n", aggregate_function="sum").sort("dia")

dia,7,8,9,10,11,12,13,14,15,16,17,18,19,20,6
i8,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1,156,2855,7654,11635,14427,21763,20673,17918,14410,8275,5072,109,0,0,0
2,254,2897,8949,12699,16027,23944,22133,18259,13334,9124,5193,916,284,8,0
3,180,3213,8059,13239,15686,24244,21816,17999,13675,8601,4033,0,14,18,0
4,280,2974,8460,13717,15076,23134,22653,18802,14523,11875,9006,6151,7510,1824,1
5,183,3243,7778,12381,14192,18631,15415,14001,9629,4668,2939,2,88,7,0
6,0,0,0,30,4,102,101,56,70,37,0,0,0,0,0
7,0,0,80,8603,21433,27170,24158,19669,20501,8523,0,0,0,0,0


Confirmamos el patron del OVA: pico en **septiembre-noviembre** y
practicamente nada el **sabado** (400 filas en dos años).

In [7]:
sab = ventas.filter(pl.col("InvoiceDate").dt.weekday() == 6).height
print(f"filas en sabado: {sab}")

filas en sabado: 400


## 3. Asociaciones — Quantity frente a Price

In [8]:
asoc = asociaciones(ventas, "Quantity", "Price")
print(f"Pearson  : {asoc['pearson']:.3f}")
print(f"Spearman : {asoc['spearman']:.3f}")

Pearson  : -0.005
Spearman : -0.428


La **Spearman** fuertemente negativa (−0,428) frente a una
**Pearson** casi nula (−0,005) es el ejemplo del OVA: el orden importa,
no el valor absoluto. En productos de regalo caros se vende poca unidad;
en los baratos muchas.

## 4. Nivel cliente — frecuencia vs monetario

In [9]:
por_cliente = ventas.group_by("Customer ID").agg(
    pl.col("Invoice").n_unique().alias("frecuencia"),
    pl.col("Importe").sum().alias("monetario"),
)
rho = por_cliente.select(pl.corr("frecuencia", "monetario", method="spearman")).item()
print(f"Spearman frecuencia-monetario: {rho:.3f}")

Spearman frecuencia-monetario: 0.858


In [10]:
por_cliente_ord = por_cliente.sort("monetario", descending=True)
n_top = int(0.2 * por_cliente_ord.height)
share = por_cliente_ord.head(n_top)["monetario"].sum() / por_cliente_ord["monetario"].sum()
print(f"participacion del 20% mayor: {share:.2%}")

participacion del 20% mayor: 77.23%


## Hallazgos para `HALLAZGOS.md`

| Hallazgo | Evidencia | Decision |
|---|---|---|
| Precio con cola larga | Cerca de Tukey = 7,50; 8,4 % la supera | Log1p en U3 |
| Cantidad con cola larga | Cerca de Tukey = 27; 6,56 % la supera | Log1p en U3 |
| Pearson ~ 0 pero Spearman −0,43 | Pearson −0,005; Spearman −0,428 | Priorizar ranks (features) |
| Concentracion fuerte en pocos clientes | 20 % superior = 77,2 % | Segmentar antes de ofertar |
| Sabado casi inexistente | 400 filas en dos años | Ignorar al modelar estacionalidad semanal |

> **Siguiente paso:** `03_variables_cliente.ipynb` — construimos RFM
> con fecha de corte, el **puente** a la Unidad 3.